# Notebook 1.2b - Annual Dispatch Reducer Prototype

This notebook is a prototype for rearchitecting Strategy 2 from Notebook 1.2.

**Goal:**
- Replace sampled weeks with a **month-by-month reducer** over the full 2025 year.
- Keep memory bounded by reducing each month immediately rather than concatenating raw annual 5-minute dispatch.
- Preserve the same core data source: `DISPATCHLOAD.INITIALMW` plus the AEMO registration workbook.

**Required data prep:**
```bash
uv run import_nem_data.py --start 2025/01/01 --end 2025/12/31 --dispatchload
```
Save `NEM Registration and Exemption List.xlsx` into `../data/nemosis_cache/`.

**Reducer design:**
- Load one month of `DISPATCHLOAD` at a time.
- Filter `INTERVENTION == 0` and shift to interval-start timestamps.
- Join only the registration columns needed for aggregation.
- Reduce immediately to `unit_day`, `unit_month`, and `region_fuel_month`.
- Derive `annual_unit` from the monthly reducer outputs.

**Output choices:**
- Keep `INITIALMW` as the source dispatch measure for comparability with Notebook 1.2.
- Clip negative dispatch to zero when aggregating generation MWh so battery charging does not subtract from generation-mix visuals.
- Do not retain raw annual interval data in memory.


In [ ]:
import calendar
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

YEAR = 2025
YEAR_MONTHS = [f'{YEAR}{month:02d}' for month in range(1, 13)]
CACHE = Path('../data/nemosis_cache')

dispatchload_by_month = {
    f.stem.split('#')[3][:6]: f
    for f in sorted(CACHE.glob('*DISPATCHLOAD*.parquet'))
}

year_dispatch_duids = (
    pl.concat(
        [
            pl.scan_parquet([dispatchload_by_month[month_key]]).select('DUID').collect()
            for month_key in YEAR_MONTHS
        ],
        how='vertical',
    )
    .unique()
)

fuel_map = {
    'brown coal': 'Coal', 'black coal': 'Coal',
    'coal seam methane': 'Coal', 'waste coal mine gas': 'Coal',
    'natural gas': 'Gas', 'natural gas / fuel oil': 'Gas',
    'natural gas / diesel': 'Gas', 'natrual gas/ diesel': 'Gas',
    'diesel': 'Gas', 'kerosene': 'Gas', 'ethane': 'Gas',
    'wind': 'Wind', 'solar': 'Solar', 'hydro': 'Hydro', 'water': 'Hydro',
    'battery': 'Battery', 'battery storage': 'Battery', 'battery / solar': 'Battery',
    'grid': 'Battery',
    'biomass': 'Biomass', 'bagasse': 'Biomass',
    'renewable/ biomass / waste': 'Biomass',
    'renewable/ biomass / waste and fossil': 'Biomass',
    'landfill methane / landfill gas': 'Biomass',
    'sewerage / waste water': 'Biomass', 'biogas - sludge': 'Biomass',
    'bagasse and diesel': 'Biomass',
}

def normalize_fuel_series(series):
    return series.fillna('').astype('string').str.strip().str.lower()

registration_pd = pd.read_excel(
    CACHE / 'NEM Registration and Exemption List.xlsx',
    sheet_name='PU and Scheduled Loads',
    engine='openpyxl',
)
registration_pd = registration_pd.assign(
    FUEL_SOURCE_PRIMARY_KEY=normalize_fuel_series(registration_pd['Fuel Source - Primary']),
    FUEL_SOURCE_DESCRIPTOR_KEY=normalize_fuel_series(registration_pd['Fuel Source - Descriptor']),
    REG_CAPACITY=lambda df: pd.to_numeric(df['Reg Cap generation (MW)'], errors='coerce'),
    REGIONID=lambda df: df['Region'],
    STATIONNAME=lambda df: df['Station Name'],
)
registration_pd['FUEL_SOURCE_KEY'] = registration_pd['FUEL_SOURCE_DESCRIPTOR_KEY'].where(
    registration_pd['FUEL_SOURCE_DESCRIPTOR_KEY'].isin(fuel_map),
    registration_pd['FUEL_SOURCE_PRIMARY_KEY'],
)
registration_pd['FUEL_TYPE'] = registration_pd['FUEL_SOURCE_KEY'].map(fuel_map).fillna('Other')
registration = (
    pl.from_pandas(
        registration_pd[['DUID', 'FUEL_TYPE', 'REG_CAPACITY', 'REGIONID', 'STATIONNAME']]
    )
    .filter(pl.col('DUID').is_not_null() & (pl.col('DUID') != '-'))
    .unique(subset=['DUID'], keep='last')
    .join(year_dispatch_duids, on='DUID', how='inner')
)

dudetail_latest = (
    pl.scan_parquet(sorted(CACHE.glob('*DUDETAIL#FILE01#*.parquet')))
    .with_columns(pl.col('EFFECTIVEDATE').str.strptime(pl.Datetime, '%Y/%m/%d %H:%M:%S'))
    .sort(['DUID', 'EFFECTIVEDATE'])
    .group_by('DUID', maintain_order=True)
    .agg([
        pl.col('REGISTEREDCAPACITY').last().cast(pl.Float64, strict=False).alias('REG_CAPACITY'),
        pl.col('DISPATCHTYPE').last().alias('FALLBACK_DISPATCHTYPE'),
    ])
    .collect()
)

dudetailsummary_latest = (
    pl.scan_parquet(sorted(CACHE.glob('*DUDETAILSUMMARY*.parquet')))
    .with_columns(pl.col('START_DATE').str.strptime(pl.Datetime, '%Y/%m/%d %H:%M:%S'))
    .sort(['DUID', 'START_DATE'])
    .group_by('DUID', maintain_order=True)
    .agg([
        pl.col('REGIONID').last().alias('REGIONID'),
        pl.col('STATIONID').last().alias('STATIONNAME'),
    ])
    .collect()
)

registration_backfill = (
    year_dispatch_duids
    .join(registration.select('DUID'), on='DUID', how='anti')
    .join(dudetail_latest, on='DUID', how='left')
    .join(dudetailsummary_latest, on='DUID', how='left')
    .with_columns([
        pl.when(pl.col('FALLBACK_DISPATCHTYPE') == 'BIDIRECTIONAL')
        .then(pl.lit('Battery'))
        .otherwise(pl.lit('Other'))
        .alias('FUEL_TYPE'),
        pl.coalesce([pl.col('STATIONNAME'), pl.col('DUID')]).alias('STATIONNAME'),
    ])
    .select(['DUID', 'FUEL_TYPE', 'REG_CAPACITY', 'REGIONID', 'STATIONNAME'])
)

registration = pl.concat([registration, registration_backfill], how='vertical_relaxed')

REGISTRATION_COLUMNS = ['DUID', 'FUEL_TYPE', 'REG_CAPACITY', 'REGIONID', 'STATIONNAME']
DISPATCH_CASTS = [
    pl.col('SETTLEMENTDATE').str.strptime(pl.Datetime, '%Y/%m/%d %H:%M:%S'),
    pl.col('INITIALMW').cast(pl.Float32),
    pl.col('INTERVENTION').cast(pl.Int8),
]

stack_order = ['Coal', 'Hydro', 'Wind', 'Solar', 'Gas', 'Battery', 'Biomass', 'Other']
stack_colors = {
    'Coal': '#4A4A4A',
    'Gas': '#FF6B6B',
    'Hydro': '#4ECDC4',
    'Wind': '#95E1D3',
    'Solar': '#FFD93D',
    'Battery': '#9D65C9',
    'Biomass': '#6BCB77',
    'Other': '#B8B8B8',
}
region_order = ['NSW1', 'QLD1', 'SA1', 'TAS1', 'VIC1']
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

def month_hours(month_key):
    year = int(month_key[:4])
    month = int(month_key[4:6])
    return calendar.monthrange(year, month)[1] * 24

def present_stack_fuels(frame):
    return [fuel for fuel in stack_order if fuel in frame.columns]

def load_dispatch_month(month_key):
    month_file = dispatchload_by_month[month_key]
    return (
        pl.scan_parquet([month_file])
        .with_columns(DISPATCH_CASTS)
        .filter(pl.col('INTERVENTION') == 0)
        .select([
            (pl.col('SETTLEMENTDATE') - pl.duration(minutes=5)).alias('SETTLEMENTDATE'),
            'DUID',
            pl.col('INITIALMW').alias('DISPATCH_MW'),
        ])
        .collect()
    )

def reduce_month(month_key):
    hours_in_month = month_hours(month_key)

    month_dispatch = load_dispatch_month(month_key)
    missing_registration = (
        month_dispatch
        .select('DUID')
        .unique()
        .join(registration.select('DUID'), on='DUID', how='anti')
        .with_columns(pl.lit(month_key).alias('YEAR_MONTH'))
    )
    month_enriched = (
        month_dispatch
        .join(registration.select(REGISTRATION_COLUMNS), on='DUID', how='left')
        .filter(pl.col('FUEL_TYPE').is_not_null())
        .with_columns([
            pl.lit(month_key).alias('YEAR_MONTH'),
            pl.lit(int(month_key[4:6])).alias('MONTH_NUM'),
            pl.col('SETTLEMENTDATE').dt.date().alias('DAY'),
            (pl.col('DISPATCH_MW').clip(lower_bound=0) / 12).alias('GEN_MWH_INTERVAL'),
        ])
    )

    unit_day = (
        month_enriched
        .group_by(['DUID', 'DAY', 'YEAR_MONTH', 'MONTH_NUM'])
        .agg([
            pl.col('GEN_MWH_INTERVAL').sum().alias('GEN_MWH_DAY'),
            pl.col('DISPATCH_MW').clip(lower_bound=0).mean().alias('AVG_MW_DAY'),
            pl.col('REGIONID').first(),
            pl.col('FUEL_TYPE').first(),
            pl.col('REG_CAPACITY').first(),
            pl.col('STATIONNAME').first(),
        ])
        .with_columns(
            pl.when(pl.col('REG_CAPACITY') > 0)
            .then(pl.col('GEN_MWH_DAY') / (pl.col('REG_CAPACITY') * 24))
            .otherwise(None)
            .alias('CF_DAY')
        )
    )

    unit_month = (
        month_enriched
        .group_by(['DUID', 'YEAR_MONTH', 'MONTH_NUM'])
        .agg([
            pl.col('GEN_MWH_INTERVAL').sum().alias('GEN_MWH_MONTH'),
            pl.col('DISPATCH_MW').clip(lower_bound=0).mean().alias('AVG_MW_MONTH'),
            pl.col('REGIONID').first(),
            pl.col('FUEL_TYPE').first(),
            pl.col('REG_CAPACITY').first(),
            pl.col('STATIONNAME').first(),
        ])
        .with_columns(
            pl.when(pl.col('REG_CAPACITY') > 0)
            .then(pl.col('GEN_MWH_MONTH') / (pl.col('REG_CAPACITY') * hours_in_month))
            .otherwise(None)
            .alias('CF_MONTH')
        )
    )

    region_fuel_month = (
        month_enriched
        .group_by(['REGIONID', 'FUEL_TYPE', 'YEAR_MONTH', 'MONTH_NUM'])
        .agg(pl.col('GEN_MWH_INTERVAL').sum().alias('GEN_MWH_MONTH'))
    )

    return unit_day, unit_month, region_fuel_month, missing_registration

print(f'DISPATCHLOAD files discovered: {len(dispatchload_by_month)}')
print('Reducer months:', YEAR_MONTHS)
print(f'Registration DUIDs covered from workbook: {registration.height - registration_backfill.height:,}')
print(f'Registration DUIDs backfilled from DUDETAIL/DUDETAILSUMMARY: {registration_backfill.height:,}')
print('Fuel types in analysis universe:')
print(registration.group_by('FUEL_TYPE').len().sort('len', descending=True))


In [ ]:
# Month-by-month reducer loop
unit_day_parts = []
unit_month_parts = []
region_fuel_month_parts = []
missing_registration_parts = []
reducer_log = []

for month_key in YEAR_MONTHS:
    print(f'Reducing {month_key}...')
    unit_day_month, unit_month_month, region_fuel_month_month, missing_registration_month = reduce_month(month_key)

    unit_day_parts.append(unit_day_month)
    unit_month_parts.append(unit_month_month)
    region_fuel_month_parts.append(region_fuel_month_month)
    missing_registration_parts.append(missing_registration_month)

    if len(missing_registration_month) > 0:
        print('  Missing registration rows:', missing_registration_month['DUID'].to_list())

    reducer_log.append({
        'YEAR_MONTH': month_key,
        'UNIT_DAY_ROWS': len(unit_day_month),
        'UNIT_MONTH_ROWS': len(unit_month_month),
        'REGION_FUEL_MONTH_ROWS': len(region_fuel_month_month),
        'MISSING_REGISTRATION_DUIDS': len(missing_registration_month),
    })

unit_day = pl.concat(unit_day_parts, how='vertical')
unit_month = pl.concat(unit_month_parts, how='vertical')
region_fuel_month = pl.concat(region_fuel_month_parts, how='vertical')
missing_registration = pl.concat(missing_registration_parts, how='vertical_relaxed')
reducer_log = pl.DataFrame(reducer_log)

print()
print('Reducer output shapes:')
print('unit_day:', unit_day.shape)
print('unit_month:', unit_month.shape)
print('region_fuel_month:', region_fuel_month.shape)
print('missing_registration:', missing_registration.shape)
print()
print('Per-month reducer log:')
print(reducer_log)
if len(missing_registration) > 0:
    print()
    print('Unmatched DUIDs that still fell out of registration coverage:')
    print(missing_registration)


In [ ]:
# Derive annual unit summaries from monthly reducers
annual_unit = (
    unit_month
    .group_by('DUID')
    .agg([
        pl.col('GEN_MWH_MONTH').sum().alias('GEN_MWH_ANNUAL'),
        pl.col('REGIONID').first(),
        pl.col('FUEL_TYPE').first(),
        pl.col('REG_CAPACITY').first(),
        pl.col('STATIONNAME').first(),
    ])
    .with_columns([
        (pl.col('GEN_MWH_ANNUAL') / 8760).alias('AVG_MW_ANNUAL'),
        pl.when(pl.col('REG_CAPACITY') > 0)
        .then(pl.col('GEN_MWH_ANNUAL') / (pl.col('REG_CAPACITY') * 8760))
        .otherwise(None)
        .alias('CF_ANNUAL'),
    ])
    .sort('GEN_MWH_ANNUAL', descending=True)
)

print('Annual reducer outputs:')
print(f'unit_day memory: {unit_day.estimated_size("mb"):.1f} MB')
print(f'unit_month memory: {unit_month.estimated_size("mb"):.1f} MB')
print(f'region_fuel_month memory: {region_fuel_month.estimated_size("mb"):.1f} MB')
print(f'annual_unit memory: {annual_unit.estimated_size("mb"):.1f} MB')
print('\nTop of annual_unit:')
print(annual_unit.head(10))


In [ ]:
# Regional fuel mix by month
fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True, sharey=True)
axes = axes.flatten()

for idx, region in enumerate(region_order):
    ax = axes[idx]
    region_month = (
        region_fuel_month
        .filter(pl.col('REGIONID') == region)
        .pivot(index='MONTH_NUM', on='FUEL_TYPE', values='GEN_MWH_MONTH', aggregate_function='sum')
        .sort('MONTH_NUM')
        .fill_null(0)
    )

    if len(region_month) == 0:
        ax.set_visible(False)
        continue

    fuels = present_stack_fuels(region_month)
    months = region_month['MONTH_NUM'].to_numpy()
    bottom = np.zeros(len(months))

    for fuel in fuels:
        values = region_month[fuel].to_numpy() / 1000
        ax.bar(months, values, bottom=bottom, color=stack_colors[fuel], width=0.8, label=fuel)
        bottom += values

    ax.set_title(region)
    ax.set_xticks(months)
    ax.set_xticklabels([month_labels[m - 1] for m in months], rotation=0)
    ax.set_ylabel('GWh')
    ax.grid(alpha=0.25, axis='y')

axes[-1].axis('off')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Fuel Type', bbox_to_anchor=(0.98, 0.98), loc='upper right')
fig.suptitle('Regional Generation Mix by Month - 2025 Reducer', y=0.995)
plt.tight_layout(rect=[0, 0, 0.92, 0.97])
plt.show()


In [ ]:
# Annual unit utilization scatter
fig, ax = plt.subplots(figsize=(12, 7))

for fuel in ['Coal', 'Gas', 'Hydro', 'Wind', 'Solar', 'Biomass']:
    subset = annual_unit.filter((pl.col('FUEL_TYPE') == fuel) & (pl.col('GEN_MWH_ANNUAL') > 0))
    if len(subset) == 0:
        continue

    ax.scatter(
        subset['REG_CAPACITY'].to_numpy(),
        subset['CF_ANNUAL'].to_numpy() * 100,
        label=fuel,
        color=stack_colors.get(fuel, '#B8B8B8'),
        alpha=0.65,
        s=55,
    )

ax.set_xlabel('Registered Capacity (MW)')
ax.set_ylabel('Annual Capacity Factor (%)')
ax.set_title('Annual Unit Utilization from Month-by-Month Reducers')
ax.set_xlim(0, None)
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Coal low-output / outage heatmap
coal_leaders = (
    annual_unit
    .filter(pl.col('FUEL_TYPE') == 'Coal')
    .sort('GEN_MWH_ANNUAL', descending=True)
    .head(15)
)

coal_duids = coal_leaders['DUID'].to_list()
all_days = unit_day.select('DAY').unique().sort('DAY')['DAY'].to_list()
coal_matrix = np.zeros((len(coal_duids), len(all_days)))

for row_idx, duid in enumerate(coal_duids):
    series = (
        unit_day
        .filter(pl.col('DUID') == duid)
        .select(['DAY', 'CF_DAY'])
        .sort('DAY')
    )
    value_map = dict(zip(series['DAY'].to_list(), series['CF_DAY'].fill_null(0).to_list()))
    for col_idx, day in enumerate(all_days):
        coal_matrix[row_idx, col_idx] = value_map.get(day, 0.0)

month_tick_positions = [idx for idx, day in enumerate(all_days) if getattr(day, 'day', None) == 1]
month_tick_labels = [month_labels[all_days[idx].month - 1] for idx in month_tick_positions]

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(coal_matrix, aspect='auto', cmap='viridis', vmin=0, vmax=1)
ax.set_yticks(range(len(coal_duids)))
ax.set_yticklabels(coal_duids)
ax.set_xticks(month_tick_positions)
ax.set_xticklabels(month_tick_labels, rotation=0)
ax.set_xlabel('Day of year')
ax.set_ylabel('Coal unit')
ax.set_title('Daily Coal Capacity Factor Heatmap - Top Annual Coal Units')
fig.colorbar(im, ax=ax, label='Daily CF')
plt.tight_layout()
plt.show()


In [ ]:
# Top generators by annual generation
top_20 = (
    annual_unit
    .head(20)
    .with_columns((pl.col('GEN_MWH_ANNUAL') / 1000).alias('GEN_GWH_ANNUAL'))
)

print('Top 20 generators by annual generation:')
print(top_20.select(['DUID', 'STATIONNAME', 'FUEL_TYPE', 'REGIONID', 'REG_CAPACITY', 'GEN_GWH_ANNUAL', 'CF_ANNUAL']))

fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(
    range(len(top_20)),
    top_20['GEN_GWH_ANNUAL'].to_numpy(),
    color=[stack_colors.get(fuel, '#B8B8B8') for fuel in top_20['FUEL_TYPE'].to_list()],
)
ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(top_20['DUID'].to_list(), fontsize=9)
ax.set_xlabel('Annual generation (GWh)')
ax.set_title('Top 20 Generators by Annual Generation - Reducer Output')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## Notes

This prototype keeps the annual analysis memory-bounded by reducing each month immediately.

If these reducer outputs prove more stable and informative than the sampled-week approach, the main Notebook 1.2 can be rearchitected so that:
- representative-week visuals stay as the high-resolution intuition layer
- annual CF, regional mix, outage screening, and top-generator rankings come from full-year monthly reducers instead of sampled weeks
